## Read the processing instructions from the configuration file

In [ ]:
import yaml
fp = "../config/sba_loans/data_processing.yml"

with open(fp, "r") as file:
    cfg_sba = yaml.safe_load(file)

    


## A brief summary of the dataset
The small business administration (SBA) makes it possible for small businesses to obtain working capital. They gaurantee borowers, lenders use this guarantee to lend money to business owners. For more information, see [this page](https://www.sba.gov/funding-programs/loans/7a-loans). The SBA reports loan performance. Most loans are paid in full, a small fraction default. We can apply machine learning to determine which loans default. Please refer to the data dictionary in the data folder for detailed description of attributes in the dataset.

In [ ]:
cfg_sba["raw_data"]

In [ ]:
fp = "../data/" + cfg_sba["raw_data"]["location"]

In [ ]:
import pandas as pd
dfr = pd.read_csv(fp)

In [ ]:
dfr.head()

In [ ]:
attr_list = dfr.columns.tolist()
pd.DataFrame({"attribute": attr_list})

In [ ]:
dfr["AsofDate"] = pd.to_datetime(dfr.AsOfDate)

In [ ]:
len(dfr["BankFDICNumber"].unique())

In [ ]:
len(dfr["BorrName"].unique())

In [ ]:
dfr["LoanStatus"].unique()

In [ ]:
len(dfr["NaicsCode"].unique())

## Exclude small number of loans with unknown business implication
There are about 3 loans which have the first disbursement date that is later than the date the loan was pain in full. This is either a data issue or, more likely, a business condition that is not

In [ ]:
dfr["FirstDisbursementDate"] = pd.to_datetime(dfr["FirstDisbursementDate"])
dfr["PaidInFullDate"] = pd.to_datetime(dfr["PaidInFullDate"])
pna = (dfr["FirstDisbursementDate"].isna()) | (dfr["FirstDisbursementDate"] > dfr["PaidInFullDate"])

to_exclude = (dfr.LoanStatus == "EXEMPT") | (dfr.LoanStatus == "CANCLD") | (dfr.LoanStatus == "COMMIT") | pna
dfr = dfr[~ to_exclude]

## Compute Initial Imbalance

In [ ]:
num_PIF = dfr[dfr.LoanStatus == "PIF"].shape[0]
num_CHGOFF = dfr[dfr.LoanStatus == "CHGOFF"].shape[0]

In [ ]:
dfr["LoanStatus"].value_counts()

In [ ]:
pct_chgoff = (num_CHGOFF/dfr.shape[0])*100
pct_pif = (num_PIF/dfr.shape[0])* 100
print(f" percent charged off is {pct_chgoff:.2f} %, percent paid in full {pct_pif:.2f}%")

In [ ]:
from datetime import datetime

def diff_month(row):
    if row["LoanStatus"] == "PIF":
        return (row["PaidInFullDate"].year - row["FirstDisbursementDate"].year) * 12 + row["PaidInFullDate"].month - row["FirstDisbursementDate"].month
    return (row["AsofDate"].year - row["FirstDisbursementDate"].year) * 12 + row["AsofDate"].month - row["FirstDisbursementDate"].month

## Create an Identifier for the Loan
A loan is one of the entity abstractions. This does not have an identifier, so we create one.

In [ ]:
dfr["LoanID"] = dfr.apply(lambda row: "Loan-" + str(row.name), axis=1)

In [ ]:
dfr["LoanID"]

## Create an attribute for number of Payments
The number of payments made with the loan is a derived attribute.

In [ ]:
dfr["NumPmtsMade"] = dfr.apply(diff_month, axis=1)

In [ ]:
dfr["NumPmtsMade"].describe()

In [ ]:
# parse configuration
entity_dict = {}
for entity_desc in cfg_sba["entities"]:
    entity_name = [*entity_desc][0]
    entity = entity_desc[entity_name]
    if entity_name not in entity_dict:
        entity_dict[entity_name] = []
    for k, v in entity.items():
        for adict in v:
            entity_dict[entity_name].append(adict["name"])

    

In [ ]:
alist = []
for key, value in entity_dict.items():
    for v in value:
        alist.append(v)

In [ ]:
dfr = dfr[alist]

In [ ]:
dfr

In [ ]:
dfr.LoanStatus.value_counts()

In [ ]:
dfr.dtypes

## Fix Missing Values

In [ ]:
def bad_FDICNumber(row):

    if pd.isna(row["BankFDICNumber"]):
        idstr = row["BankName"][:10] + "-" + str(row["BankZip"])
        row["BankFDICNumber"] = idstr
    if isinstance(row["BankFDICNumber"], float):
        row["BankFDICNumber"] = int(row["BankFDICNumber"])
    return row["BankFDICNumber"]
dfr["BankFDICNumber"] = dfr.apply(bad_FDICNumber, axis=1)

In [ ]:
dfr.isna().sum().sum()

In [ ]:
null_FDIC = dfr.BankFDICNumber.isna()
dfr[null_FDIC]

## Correct Data Types

In [ ]:
dfr.loc[:, "BankFDICNumber"] = dfr["BankFDICNumber"].astype(str)
dfr.loc[:, "BankZip"] = dfr["BankZip"].astype(str)
dfr.loc[:, "BorrZip"] = dfr["BorrZip"].astype(str)
dfr.loc[:, "NaicsCode"] = dfr["NaicsCode"].astype(str)

In [ ]:
dfr.dtypes

## Fix Support for Codes
Some of the Zip Codes and NAICS codes have insufficient support. We can fix this by considering only the first few characters of the code. These codes are hierarchical, so doing this merges hierarchically

In [ ]:
dfr["BorrZip"] = dfr.BorrZip.str[:2] + 3*"X" 

## Verify Support
Verify the zip codes and NAICS codes have value counts of at least 5 per category

In [ ]:
dfr["BorrZip"].value_counts()

## Fix Support for Cities
Many Borrower and Lender cities have only one or two entries. Collectively, an insufficient support category is created and these loans are bucketed there.

In [ ]:
dfr["BankCity"].value_counts()

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
dfr["BankZip"] = dfr.BankZip.str[:3] + 2*"X" 
dfr["BankZip"].value_counts()

In [ ]:
insuff_supp_bank_city = [index for index, value in dfr["BankCity"].value_counts().items() if value < 5]
insuff_supp_borr_city = [index for index, value in dfr["BorrCity"].value_counts().items() if value < 5]

In [ ]:
recode_bank_city = dfr["BankCity"].isin(insuff_supp_bank_city)

recode_borr_city = dfr["BorrCity"].isin(insuff_supp_borr_city)



In [ ]:
dfr.loc[recode_borr_city, "BorrCity"] = "XXXX"
dfr.loc[recode_bank_city, "BankCity"] = "XXXX"

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
dfr["BankCity"].value_counts()

In [ ]:
#drop_insuff_supp = dfr.BorrZip == "69XXX"
#dfr = dfr[~drop_insuff_supp]

In [ ]:
dfr["BankZip"].value_counts()

In [ ]:
drop_insuff_supp = dfr.BankZip == "51XXX"
dfr = dfr[~drop_insuff_supp]

In [ ]:
dfr.loc[:, "NaicsCode"] = dfr.NaicsCode.str[:2] + 4*"X" 

In [ ]:
dfr.NaicsCode.value_counts()

In [ ]:
insuff_supp_borr_zip = [index for index, value in dfr["BorrZip"].value_counts().items() if value < 5]

In [ ]:
insuff_supp_borr_zip

## Use the Single Label Criterion to identify easy classification regions
This a version of the OneR Classifier. For the Naics codes and Borrower cities, look for category levels that have only one label associated with them. This is a very easy classifier to implement. All you need to do is check if a borrower belongs to a particular City and then if they do, just classify them as good borrowers

In [ ]:
df_chgoff = dfr[dfr.LoanStatus == "CHGOFF"]
df_PIF = dfr[dfr.LoanStatus == "PIF"]

In [ ]:
chgOff_NaicsCode =  set(df_chgoff.NaicsCode.unique())
PIF_NaicsCode = set(df_PIF.NaicsCode.unique())

In [ ]:
good_NaicsCode = PIF_NaicsCode.difference(chgOff_NaicsCode)
bad_NaicsCode = chgOff_NaicsCode.difference(PIF_NaicsCode)
one_label_NaicsCode = good_NaicsCode.union(bad_NaicsCode)
is_good_NaicsCode = dfr["NaicsCode"].isin(one_label_NaicsCode)

In [ ]:
bad_NaicsCode

In [ ]:
dfr = dfr[~is_good_NaicsCode]
dfr["LoanStatus"].value_counts()

In [ ]:
chgOff_BankCity = set(df_chgoff.BankCity.unique())
PIF_BankCity = set(df_PIF.BankCity.unique())
good_BankCity = PIF_BankCity.difference(chgOff_BankCity)
bad_BankCity = chgOff_BankCity.difference(PIF_BankCity)
one_label_BankCity = good_BankCity.union(bad_BankCity)
is_good_BankCity = dfr["BankCity"].isin(one_label_BankCity)

In [ ]:
bad_BankCity

In [ ]:
dfr = dfr[~is_good_BankCity]
dfr["LoanStatus"].value_counts()

In [ ]:
chgOff_BorrCity = set(df_chgoff.BorrCity.unique())
PIF_BorrCity = set(df_PIF.BorrCity.unique())
good_BorrCity = PIF_BorrCity.difference(chgOff_BorrCity)
bad_BorrCity = chgOff_BorrCity.difference(PIF_BorrCity)
one_label_BorrCity = good_BorrCity.union(bad_BorrCity)
is_good_BorrCity = dfr["BorrCity"].isin(good_BorrCity)
dfgbc = dfr[is_good_BorrCity]
dfr = dfr[~is_good_BorrCity]


## Re-evaluate Support after removing the good borrowers from the data

In [ ]:
att_list = ["BorrCity", "BorrZip", "BankCity", "BankZip"]
for a in att_list:
    insuff_supp = dfr[a].value_counts() < 15
    bad_vals = [index for index, value in insuff_supp.items() if value]
    dfr = dfr[~dfr[a].isin(bad_vals)]

In [ ]:
dfr["BorrZip"].value_counts()

In [ ]:
filter_69xxx = dfr["BorrZip"] == "69XXX"

In [ ]:
filter_outlier_states = dfr["BorrState"].isin(["WY","VI"])

In [ ]:
dfr = dfr[~filter_outlier_states]

In [ ]:
dfr = dfr[~filter_69xxx]

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_stage1.csv"
dfr.to_csv(fp, index=False)

## Create Good Borrower States Graph

Networkx is used to create this graph. The example used here is the state of California. There are several cities in California that have no charge off's. The circular layout is used to plot the graph. In a circular layout scheme, the cities that have no default in california are laid out in a circle and connected to the state of "CA". The cities are in the circumference of the circle and they connect by radial edges to the state node of California.  

In [ ]:

borr_att = entity_dict["borrower"]

In [ ]:
dfgbc = dfgbc[borr_att]

In [ ]:
dfgbc.BorrState.unique()

In [ ]:
df_graph = dfgbc.groupby("BorrState")["BorrCity"].value_counts(normalize=True).reset_index()

In [ ]:
df_graph = df_graph.groupby("BorrState")

In [ ]:
import networkx as nx
G = nx.Graph()
edge_list = []
for state, group in df_graph:
    G.add_node(state)
    G.nodes[state]["name"] = state
    G.nodes[state]["type"] = "STATE"
    
    for idx, row in group.iterrows():
        city = row["BorrCity"]
        wt = round(row["proportion"],3)
        G.add_node(city)
        G.nodes[city]["name"] = row["BorrCity"]
        G.nodes[city]["type"] = "CITY"
        edge_list.append((state, city, wt))

G.add_weighted_edges_from(edge_list)

In [ ]:
dfgbc

In [ ]:
good_cities_in = G.neighbors("CA")
el = [("CA", c) for c in good_cities_in]

In [ ]:
import matplotlib.pyplot as plt

# Create a graph
cagc = nx.Graph()
cagc.add_edges_from(el)
# Generate circular layout positions
pos = nx.circular_layout(cagc)

# Draw the graph with labels
plt.figure(figsize=(12, 8)) # Optional: set figure size
nx.draw_networkx(cagc, pos, with_labels=True, node_color='skyblue', node_size=10, font_size=8, edge_color='gray')

# Display the plot
plt.title("Good Borrower Cities in CA")
plt.axis('off') # Hide axes
plt.show()

In [ ]:
dfr["LoanStatus"].value_counts()

In [ ]:
chgOff_ProjectCounty = set(df_chgoff.ProjectCounty.unique())
PIF_ProjectCounty = set(df_PIF.ProjectCounty.unique())
good_ProjectCounty = PIF_ProjectCounty.difference(chgOff_ProjectCounty)
good_ProjectCounty = good_ProjectCounty.union(chgOff_ProjectCounty.difference(PIF_ProjectCounty))
is_good_ProjectCounty = dfr["ProjectCounty"].isin(good_ProjectCounty)
dfr = dfr[~is_good_ProjectCounty]
dfr["LoanStatus"].value_counts()

## Note About Change in Imbalance
As you can see below, removing the homogeneous label regions, nearly doubles the percentage of charge off loans in the dataset. This dataset can now be used for graph machine learning. The entity definitions and the relations definitions can be used to create a heterogeneous graph and an appropriate graph machine learning library, DGL, PytorchGeometric etc, can be used to predict if a loan is going to be charged off.

In [ ]:
num_PIF = dfr[dfr.LoanStatus == "PIF"].shape[0]
num_CHGOFF = dfr[dfr.LoanStatus == "CHGOFF"].shape[0]
pct_chgoff = (num_CHGOFF/dfr.shape[0])*100
pct_pif = (num_PIF/dfr.shape[0])* 100
print(f" percent charged off is {pct_chgoff:.2f} %, percent paid in full {pct_pif:.2f}%")

In [ ]:
fp = "../data/olist_prepared/sba_loans_stage_1.csv"
dfr.to_csv(fp, index=False)

In [ ]:
cfg_sba["relations"]

In [ ]:
borr_bank_relation_att = ["BorrName", "BankFDICNumber"]
bank_loan_relation_att = ["BankFDICNumber", "LoanID"]

In [ ]:
borr_bank_rel_df = dfr[borr_bank_relation_att]
bank_loan_rel_df = dfr[bank_loan_relation_att]

In [ ]:
borr_bank_rel_df

In [ ]:
bank_loan_rel_df 

In [ ]:
cfg_sba["relations"].keys()

In [ ]:
entities = {}
edge_defn = []
for rel, defn in cfg_sba["relations"].items():
    edge_defn.append((defn["from"], defn["to"]))

In [ ]:
edge_defn

In [ ]:
import pydot
schema_graph = pydot.Dot(graph_type="graph")

for e in edge_defn:
    schema_graph.add_edge(pydot.Edge(e[0], e[1]))

In [ ]:
fp = "../images/sba_7aloans_graph_schema.png"
schema_graph.write_png(fp)

<div> <img src="../images/sba_7aloans_graph_schema.png"> </div>